# 3. Accelerometry (optional)

**Step 3 of 4**: codebook -> download & parse -> accelerometry (optional) -> analysis.

Minute-by-minute motion sensor data. NHANES measured it twice: a hip monitor in 2003-2006 and a wrist
monitor in 2011-2014. One row per participant, one column per minute.

## Download the 2026-pre-parsed step counts

pyNHANES publishes the parsed arrays as GitHub release assets. This notebook uses `nhanes_steps.npz` (20MB, 7430 participants of 2005-2006).

The other two are the same shape and can be read the same way:
`nhanes_counts.npz` (89MB, 14631 participants of 2003-2006) and
`nhanes_triax.npz` (310MB, 14693 participants of 2011-2014).

In [ ]:
!mkdir -p ./NPZ ./CSV
# downloaded once - delete the file to fetch a fresh copy
!test -f ./NPZ/nhanes_steps.npz || curl -sSL -o ./NPZ/nhanes_steps.npz https://github.com/timpyrkov/pynhanes/releases/download/data-v1/nhanes_steps.npz

# the parsed table of notebook 2, if it is not beside us already
!test -f ./CSV/nhanes_userdata.csv.gz || curl -sSL -o ./CSV/nhanes_userdata.csv.gz https://github.com/timpyrkov/pynhanes/releases/download/data-v1/nhanes_userdata.csv.gz

## Alternatively: download the raw accelerometry and parse it yourself

Uncomment and run it only if you want the arrays rebuilt from the NHANES files.

`PAXRAW` of 2005-2006 is 0.44 GB zipped and 2.8 GB unpacked; `PAXMIN` of 2011-2014 is
16.3 GB and comes from the slow `ftp.cdc.gov`. Parsing reads them in chunks (~1.5 GB of
memory): 40 seconds for 2003-2006, 2 minutes for 2011-2014.

In [ ]:
# !pynhanes-downloader -o ./XPT -a 2005
# !pynhanes-activity -i ./XPT -o ./NPZ -c ./CSV/nhanes_activity.csv -e paxraw

## What is in the file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pynhanes

npz = np.load("./NPZ/nhanes_steps.npz")
for key in npz:
    value = npz[key]
    print(f"{key:24s} {str(value.shape):16s} {value.dtype}")

| key | meaning |
|---|---|
| `userid` | `SEQN`, the NHANES participant number |
| `steps` | Steps in that minute, clipped to 0-255 to fit one byte |
| `weekday` | 1 Monday ... 7 Sunday, 0 missing |
| `status` | What the minute was, see below |
| `Rolled by index` | By how many minutes the row was rolled |
| `ISO weekday` | Whether weekdays are 1 Monday, not 1 Sunday as NHANES numbers them |
| `Status keys` | The labels to decode `status` |

**Column 0 is midnight of the first day of wear**, and the 7-day arrays are rolled so that
column 0 is **Monday 00:00**. Every array of one participant is rolled by the same number of
minutes, so `steps[i]`, `weekday[i]` and `status[i]` stay aligned.

## Status codes

Only 2011-2014 has the wake/sleep/non-wear prediction. For 2003-2006 a minute either holds
data (`4` Unknown) or it does not (`0` Missing) - which is how a minute with no data is told
apart from a minute of perfect stillness.

In [ ]:
status = npz["status"]
for value, count in zip(*np.unique(status, return_counts=True)):
    print(f"{value}  {pynhanes.activity.decode_status(value):12s} {count:>12,}  "
          f"{100 * count / status.size:5.1f}%")

## The average week

Minutes with no data are left out rather than counted as zero movement.

In [ ]:
steps = npz["steps"].astype(float)
steps[status == 0] = np.nan
average = np.nanmean(steps, axis=0)

plt.figure(figsize=(14, 4), facecolor="white")
plt.plot(np.arange(len(average)) / 1440, average, lw=0.7)
plt.xticks(np.arange(7) + 0.5, ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.xlim(0, 7)
plt.ylabel("Steps per minute")
plt.title(f"Average week, {len(npz['userid'])} participants of 2005-2006")
plt.grid(axis="x", alpha=0.3)
plt.show()

## Reading it together with the questionnaire

`pynhanes.NhanesLoader` attaches whichever `.npz` files are in the folder to the parsed
table, so the accelerometry and the answers line up on `SEQN`. See
[notebook 4](plot_analysis.ipynb).

In [ ]:
nhanes = pynhanes.NhanesLoader(path_csv="./CSV/nhanes_userdata.csv.gz",
                               path_npz="./NPZ/", verbose=True)
print(nhanes.x.shape)